# Card-Interpretation Evals

Runs the **production agent harness** (`agent.runtime.run_agent`) over a benchmark and measures the metrics that matter: tool calls, answer similarity to canonical, executability, the *did-something* (no-op) rate, cost, and latency.

The harness reproduces real play: a live parity `GameState`, an actor who is **not** the card's author, the full production toolbox (optionally filtered), the honored caps, and vision when enabled.

**Setup:** `uv sync --group evals`, then select this repo's venv as the kernel. An LLM gateway must be configured (`LLM_API_KEY` / `LLM_BASE_URL`) — the agent and the judge both call it.

Workflow: **1) Configure -> 2) Run (persists to `data/eval/runs/`) -> 3) Load history -> 4) Compare with charts.**

In [ ]:
import logging
import sys
from datetime import datetime
from pathlib import Path

# Make src/ importable from scripts/.
sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt

from config import EVAL_BENCHMARKS, EVAL_MODEL_PRICES
from evals import store, viz
from evals.runner import EvalConfig, available_tool_names, run_benchmark

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("evals.runner").setLevel(logging.INFO)

print("Benchmarks:", list(EVAL_BENCHMARKS))
print("Tools available to toggle:", available_tool_names())
print("Known model prices (USD / 1M tok):", EVAL_MODEL_PRICES)

## 1. Configure a run

Edit `EvalConfig` to change any knob. Everything is optional except `benchmark`.

| Knob | Meaning |
|---|---|
| `benchmark` | `seed` / `eval` / `eval_hard` / `real` |
| `model_name` | chat model id; `None` = configured `LLM_CHAT_MODEL` |
| `enabled_tools` | `None` = full production toolbox; a `frozenset` of names to filter; `frozenset()` = no tools |
| `max_tool_calls` | step cap (prod default 24); `None` = default |
| `timeout` | wall-clock seconds per card; `None` = default (600) |
| `vision` | attach card art / alt_text as an image (needs a vision model) |
| `n_samples` | samples per card; `>1` reports consistency (stochastic reliability) |
| `sample_size` | cap cards per benchmark (`None` = all; **required for `real`** — 698 cards) |
| `use_judge` | run the LLM judge (costs money) vs deterministic scorers only |
| `label` | short name shown in charts |

In [ ]:
config = EvalConfig(
    benchmark="eval",
    model_name=None,  # e.g. "gpt-5.4-mini" or "gpt-5.4"
    enabled_tools=None,  # e.g. frozenset({"card_rag", "dry_run_effect"})
    max_tool_calls=None,  # e.g. 12
    timeout=None,
    vision=False,
    n_samples=1,
    sample_size=3,  # small while iterating; set None for the full set
    use_judge=True,
    label="baseline",
)

# Optionally override prices at runtime via dataclasses.replace:
#   from dataclasses import replace
#   config = replace(config, prices={"gpt-5.4-mini": {"input": 0.25, "output": 2.0}})
config

## 2. Run it (and persist)

Serial — expect roughly (cards x n_samples x per-card latency). Each run is written to `data/eval/runs/<timestamp>_<benchmark>_<model>.json` so it survives kernel restarts and can be compared later.

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
run = run_benchmark(config, timestamp=timestamp)
path = store.save_run(run, timestamp=timestamp)
print("saved:", path)

summary = run.aggregate()
viz.summary_table([summary])

## 3. Load run history

Pick which persisted runs to compare. By default this loads **every** run on disk (chronological); slice the list to compare a specific subset.

In [ ]:
payloads = store.load_runs()  # or store.load_runs([Path("data/eval/runs/....json"), ...])
summaries = [p["summary"] for p in payloads]
print(f"{len(summaries)} run(s) loaded")
viz.summary_table(summaries)

## 4. Compare runs

Runs keep a fixed color across every chart. One measure per axis; lower is better for tool-calls/cost/latency, higher for the quality metrics.

In [ ]:
viz.plot_quality(summaries)
plt.tight_layout()
plt.show()

In [ ]:
viz.plot_efficiency(summaries)
plt.tight_layout()
plt.show()

In [ ]:
viz.plot_tool_usage(summaries)
plt.tight_layout()
plt.show()

In [ ]:
viz.plot_cost_vs_quality(summaries, quality_key="intent_match")
plt.tight_layout()
plt.show()

## 5. Drill down: worst cards

The lowest-scoring cards for a chosen metric in a single run, with the judge's reason — where to look when a metric drops.

In [ ]:
viz.worst_cards(payloads[-1], metric="executability", n=10)